In [1]:
%pip install -qU langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -U ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="erojas_250000003051/ollamacollab:latest",
    temperature=0,
    # other params...
)

In [3]:
def prompt_func(data):
    text = data["text"]
    image = data["image"]

    image_part = {
        "type": "image_url",
        "image_url": f"data:image/jpeg;base64,{image}",
    }

    content_parts = []

    text_part = {"type": "text", "text": text}

    content_parts.append(image_part)
    content_parts.append(text_part)

    return [HumanMessage(content=content_parts)]

In [14]:
from PIL import Image
import os

# Convert the WEBP image to standard JPEG
file_path = "sample_essay.jpg"
output_path = "sample_essay_converted.jpg"

print(f"Converting {file_path} to standard JPEG...")
img = Image.open(file_path)
print(f"Original format: {img.format}, mode: {img.mode}, size: {img.size}")

# Convert to RGB if necessary (for JPEG compatibility)
if img.mode in ("RGBA", "P"):
    rgb_img = Image.new("RGB", img.size, (255, 255, 255))
    rgb_img.paste(img, mask=img.split()[-1] if img.mode == "RGBA" else None)
    img = rgb_img

# Save as JPEG
img.save(output_path, "JPEG", quality=95)
print(f"✓ Converted and saved to {output_path}")
print(f"  New file size: {os.path.getsize(output_path)} bytes")


Converting sample_essay.jpg to standard JPEG...
Original format: WEBP, mode: RGB, size: (2393, 3249)
✓ Converted and saved to sample_essay_converted.jpg
  New file size: 3459435 bytes


In [ ]:
import base64
import requests

# Read the converted JPEG image
file_path = "sample_essay_converted.jpg"

def encode_image_to_base64(image_path):
    """Encodes an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        encoded = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded

image_b64 = encode_image_to_base64(file_path)
print(f"Image encoded ({len(image_b64)} characters)")

prompt_text = """Evaluate the attached essay based on the following expectations: 
1. Talk about cultural practices of a particular country.
2. Talk about the issues in the cultural practices.
3. Provide your own opinion on the matter.
Use the rubric below to give a score:
90-100% (A): Exceptional work; demonstrates mastery of the assignment's objectives, using excellent grammar.
80-89% (B): Solid work; demonstrates a good understanding and fulfills most requirements, with good grammar.
70-79% (C): Satisfactory work; demonstrates an adequate understanding but may have some weaknesses, with some grammatical problems.
60-69% (D): Below average work; demonstrates a limited understanding and has significant weaknesses, with unacceptable grammatical errors.
Below 60% (F): Unsatisfactory work; fails to meet the assignment's objectives; very problematic sentences unacceptable grammatical errors."""

OLLAMA_ENDPOINT = "http://localhost:11434/api/generate"
MODEL_NAME = "llava:latest"

payload = {
    "model": MODEL_NAME,
    "prompt": prompt_text,
    "images": [image_b64],
    "stream": False
}

try:
    print(f"Sending request to Ollama using {MODEL_NAME}...")
    response = requests.post(OLLAMA_ENDPOINT, json=payload, timeout=120)
    response.raise_for_status()
    result = response.json()
    print("\n" + "="*70)
    print("ESSAY EVALUATION RESULT:")
    print("="*70)
    print(result.get("response", "No response text returned."))
except requests.RequestException as e:
    print(f"Request failed: {e}")
    print(f"Status code: {response.status_code if 'response' in locals() else 'N/A'}")
    if 'response' in locals() and response.text:
        print(f"Response: {response.text}")


Image encoded (2400940 characters)
Sending request to Ollama using llava:latest...
Request failed: 500 Server Error: Internal Server Error for url: http://localhost:11434/api/generate


In [12]:
import requests
import json

print("Pulling llava vision model...")
response = requests.post(
    "http://localhost:11434/api/pull",
    json={"name": "llava:latest"},
    stream=True
)

for line in response.iter_lines():
    if line:
        data = json.loads(line.decode() if isinstance(line, bytes) else line)
        status = data.get("status", "")
        if status:
            print(status, end="\r")

print("\nllava model pull complete!")


Pulling llava vision model...
success manifest digest
llava model pull complete!
